# Main analysis - Data quality, EDA & KPI business

## Libraries

In [1]:
#Importando as bibliotecas necessárias
import pandas as pd

## Global variables & functions

In [ ]:
#Importando arquivos do diretório Global
#Variavies
import sys
sys.path.append("../")

from Global.variables import (
    OLD_DATASET_DIR,
    NEW_DATASET_DIR,
    OUTPUT_FILE,
    DB_CONFIG
)
#Funções
import Global.functions as func

## Old dataset treatment

### Dataset overview

In [ ]:
#Chamando a função para mostrar as colunas dos arquivos CSV na pasta "Dataset"
func.show_columns(OLD_DATASET_DIR)

Arquivos encontrados: 7

Arquivo: agencias.csv
Quantidade de colunas: 7
------------------------------------------------------------
- cod_agencia
- nome
- endereco
- cidade
- uf
- data_abertura
- tipo_agencia

Arquivo: clientes.csv
Quantidade de colunas: 10
------------------------------------------------------------
- cod_cliente
- primeiro_nome
- ultimo_nome
- email
- tipo_cliente
- data_inclusao
- cpfcnpj
- data_nascimento
- endereco
- cep

Arquivo: colaborador_agencia.csv
Quantidade de colunas: 2
------------------------------------------------------------
- cod_colaborador
- cod_agencia

Arquivo: colaboradores.csv
Quantidade de colunas: 8
------------------------------------------------------------
- cod_colaborador
- primeiro_nome
- ultimo_nome
- email
- cpf
- data_nascimento
- endereco
- cep

Arquivo: contas.csv
Quantidade de colunas: 9
------------------------------------------------------------
- num_conta
- cod_cliente
- cod_agencia
- cod_colaborador
- tipo_conta
- data_aber

### Exploration of dataset - Quantity

In [ ]:
#Função para carregar todos os arquivos CSVs da pasta "Old_Dataset" e armazenar em um dicionário para iniciarmos a análise exploratória dos dados.
old_datasets = func.load_csv_files(OLD_DATASET_DIR)

for name, df in old_datasets.items():
    print(f"{name}: {df.shape}")

agencias: (10, 7)
clientes: (998, 10)
colaborador_agencia: (100, 2)
colaboradores: (100, 8)
contas: (999, 9)
propostas_credito: (2000, 12)
transacoes: (71999, 5)


In [ ]:
#Observações:
    #998 clientes -> Valor base.
    #999 contas -> Necessário verificar, pois potencialmente existe cliente com mais de uma conta, ou uma conta sem correspondência.
    #2000 propostas de crédito -> um cliente pode ter várias propostas.
    #71999 transações -> várias transações por conta.

### Exploration of dataset - Collumns types

In [6]:
#Função para inspecionar os tipos de dados de todas as tabelas carregadas
data_types = func.inspect_data_types(old_datasets)

for dataset_name, df_types in data_types.items():
    print(f"\n{'=' * 50}")
    print(dataset_name)
    print(f"{'=' * 50}")
    print(df_types.to_string(index=False))


agencias
       column data_type
  cod_agencia     int64
         nome       str
     endereco       str
       cidade       str
           uf       str
data_abertura       str
 tipo_agencia       str

clientes
         column data_type
    cod_cliente     int64
  primeiro_nome       str
    ultimo_nome       str
          email       str
   tipo_cliente       str
  data_inclusao       str
        cpfcnpj       str
data_nascimento       str
       endereco       str
            cep       str

colaborador_agencia
         column data_type
cod_colaborador     int64
    cod_agencia     int64

colaboradores
         column data_type
cod_colaborador     int64
  primeiro_nome       str
    ultimo_nome       str
          email       str
            cpf       str
data_nascimento       str
       endereco       str
            cep       str

contas
                column data_type
             num_conta     int64
           cod_cliente     int64
           cod_agencia     int64
       cod_col

In [7]:
#Observações:
    #Datas -> str -> será necessário tratar para datetime
    #Códigos -> int64 -> vale a pena verificar se não existem códigos com zeros à esquerda, que podem ser perdidos ao converter para int64.
    #Valores monetários -> float64
    #Quantidade de parcelas/carência -> int64
    #Taxa de juros -> float64

In [8]:
#Analisando o cabeçalho dos datasets para verificar se os dados sofreram mudança quando foi convertido em dataframe
for dataset_name, df in old_datasets.items():
    print(f"\n{dataset_name}")
    print(df.head().to_string(index=False))


agencias
 cod_agencia             nome                                                            endereco    cidade uf data_abertura tipo_agencia
           7  Agência Digital     Av. Paulista, 1436 - Cerqueira César, São Paulo - SP, 01310-916 São Paulo SP    2015-08-01      Digital
           1   Agência Matriz     Av. Paulista, 1436 - Cerqueira César, São Paulo - SP, 01310-916 São Paulo SP    2010-01-01       Física
           2  Agência Tatuapé       Praça Sílvio Romero, 158 - Tatuapé, São Paulo - SP, 03323-000 São Paulo SP    2010-06-14       Física
           3 Agência Campinas  Av. Francisco Glicério, 895 - Vila Lidia, Campinas - SP, 13012-000  Campinas SP    2012-03-04       Física
           4   Agência Osasco Av. Antônio Carlos Costa, 1000 - Bela Vista, Osasco - SP, 06053-014    Osasco SP    2013-11-06       Física

clientes
 cod_cliente primeiro_nome ultimo_nome                        email tipo_cliente           data_inclusao        cpfcnpj data_nascimento                 

In [ ]:
#Observações:
    #Nenhum zero a esquerda foi desconsiderado -> Excelente.
    #Valores monetários estão com o ponto como separador decimal -> Excelente.
    #Valores das transações estão negativos, o que é esperado, pois representam saídas de dinheiro da conta -> Excelente.
    #Há colunas de datas no formato YYYY-MM-DD com ou sem o horario UTC -> Necesário averiguar.

### Exploration of dataset null values

In [10]:
#Função para inspecionar os valores nulos de todas as tabelas carregadas
missing_values = func.inspect_missing_values(old_datasets)

for dataset_name, df_missing in missing_values.items():
    print(f"\n{'=' * 50}")
    print(dataset_name)
    print(f"{'=' * 50}")

    print(
        df_missing[
            df_missing["missing_count"] > 0
        ].to_string(index=False)
    )


agencias
Empty DataFrame
Columns: [column, missing_count, missing_percentage]
Index: []

clientes
Empty DataFrame
Columns: [column, missing_count, missing_percentage]
Index: []

colaborador_agencia
Empty DataFrame
Columns: [column, missing_count, missing_percentage]
Index: []

colaboradores
Empty DataFrame
Columns: [column, missing_count, missing_percentage]
Index: []

contas
Empty DataFrame
Columns: [column, missing_count, missing_percentage]
Index: []

propostas_credito
Empty DataFrame
Columns: [column, missing_count, missing_percentage]
Index: []

transacoes
Empty DataFrame
Columns: [column, missing_count, missing_percentage]
Index: []


In [ ]:
#Observações:
    #Nenhum valor nulo encontrado -> Excelente.

In [14]:
new_datasets = old_datasets.copy()

## New dataset loading

### New dataset generation

In [ ]:
#Chamando função principal para gerar os novos datasets CSVs tratados a partir dos arquivos CSVs antigos
func.save_csv_files(new_datasets, NEW_DATASET_DIR)

### Schema generation

In [ ]:
#Chamando função principal para gerar o schema SQL a partir dos arquivos CSVs novos tratados
func.generated_schema(NEW_DATASET_DIR, OUTPUT_FILE)

### PostgreSQL loading

In [ ]:
#Chamando função principal para gerar o INSERT SQL a partir dos arquivos CSVs novos tratados
#func.load_all_csvs_to_postgres(NEW_DATASET_DIR, DB_CONFIG)